## Método de Región de Confianza aplicado a SINGX

### Idea básica

En cada iteración $k$ se construye un modelo cuadrático local de la función objetivo:
$$m_k(p)=f(x_k)+g_k^t p+\tfrac{1}{2}p^t B_k p,$$
donde $g_k=\nabla f(x_k)$ y $B_k=\nabla^2 f(x_k)$. El paso $p_k$ se obtiene resolviendo el subproblema de región de confianza:
$$\min_{p\in\mathbb{R}^n}\ m_k(p)\quad \text{s.a.}\quad \|p\|\le\Delta_k,$$
con $\Delta_k>0$ el radio de la región. Una vez calculado $p_k$, se evalúa el cociente de reducción real vs. predicha:
$$\rho_k=\frac{f(x_k)-f(x_k+p_k)}{m_k(0)-m_k(p_k)}.$$
Si $\rho_k>0.75$, el modelo es confiable y se expande $\Delta_{k+1}=2\Delta_k$; si $\rho_k<0.25$, se contrae $\Delta_{k+1}=\Delta_k/4$; si $\rho_k\le 0$, se rechaza el paso ($x_{k+1}=x_k$).

### Aplicación a SINGX

- **Suavidad:** $f\in C^\infty(\mathbb{R}^n)$, el modelo cuadrático es exacto hasta segundo orden.
- **Hessiana analítica:** $\nabla_{y^{(j)}}^2 f=A^t\,\mathrm{diag}(2,\,10,\,12u_3^2,\,120u_4^2)\,A$, bloque diagonal.
- **Singularidad:** en $x^*=0$, $\nabla^2 f(x^*)$ es singular (rango $2m<n$). La región de confianza **no necesita invertir la Hessiana**: el subproblema restringido $\|p\|\le\Delta_k$ siempre tiene solución única y bien definida, incluso si $\nabla^2 f(x_k)$ es singular o indefinida.

### Subproblema: paso de Cauchy

- **Paso de Cauchy:** $p_k^C=-\alpha_k^C g_k$, con $\alpha_k^C=\min\!\left(\frac{\|g_k\|^2}{g_k^t B_k g_k},\,\frac{\Delta_k}{\|g_k\|}\right)$. Garantiza descenso suficiente pero solo convergencia lineal.

In [1]:
# Imports
import sys
from pathlib import Path

# Sube un nivel desde la carpeta del notebook hasta la raíz del repositorio
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from funciones import singx_f, singx_grad, singx_hess, trust_region_cauchy

### Ejemplo de una iteración ($n=4$, $x_0=(1,1,1,1)^t$)

- $f(x_0)=122$, $\nabla f(x_0)=(22,216,8,0)^t$, $\|\nabla f(x_0)\|\approx 217.26$.
- $\nabla^2 f(x_0)$ es singular (rango 3); el paso de Newton no está bien definido, pero el paso de Cauchy sí.
- Con $\Delta_0=1$, el paso de Cauchy es $p_0^C\approx-\Delta_0\,g_0/\|g_0\|\approx(-0.101,-0.994,-0.037,0)^t$.
- Se calcula $\rho_0$ y se ajusta $\Delta_1$.

### ¿Encuentra solución el método de Región de Confianza?

**Sí, con convergencia global.** A diferencia del método de Newton puro (que falla en $x_0=(1,1,1,1)$ por la singularidad de la Hessiana), la región de confianza **converge desde ese punto** porque no necesita invertir $\nabla^2 f(x_0)$: el paso de Cauchy está siempre bien definido. El radio $\Delta_k$ se adapta y el método es globalmente robusto.

**Valor del mínimo encontrado:** el mínimo global de SINGX es
$$x^*=(0,0,\dots,0)^t,\qquad f(x^*)=0.$$
Con el **paso de Cauchy**, la convergencia es **lineal** y se alcanza $f(x_k)\approx 10^{-8}$–$10^{-10}$ en $x_k\approx 0$ (el criterio de parada es $\|\nabla f(x_k)\|<10^{-8}$). Con el **paso Dogleg** o una resolución precisa del subproblema (GC-Steihaug), la convergencia cerca de $x^*$ es **superlineal/cuadrática** y se alcanza $f(x_k)\approx 10^{-14}$–$10^{-16}$ antes de que el redondeo de máquina imponga un límite. En ambos casos, el valor reportado en el óptimo es $f\approx 0$ (numéricamente indistinguible de cero).

In [2]:
# ----- Ejecución: Región de Confianza (Cauchy) sobre SINGX, n=4, x0=(1,1,1,1) -----
x0 = np.array([1., 1., 1., 1.])

print(f"f(x0)       = {singx_f(x0):.4f}")        # esperado: 122.0000
print(f"||grad(x0)||= {np.linalg.norm(singx_grad(x0)):.4f}")  # esperado: ≈ 217.2630
print(f"rang(H(x0)) = {np.linalg.matrix_rank(singx_hess(x0))}")  # esperado: 3 (singular)
print()

x_opt, hist = trust_region_cauchy(singx_f, singx_grad, singx_hess, x0,
                                   delta0=1.0, tol=1e-8, max_iter=500,
                                   verbose=True)

print(f"\n--- Resultado final ---")
print(f"x_opt       = {x_opt}")                 # esperado: → (0,0,0,0)
print(f"f(x_opt)    = {singx_f(x_opt):.3e}")   # esperado: → 0
print(f"||grad||    = {np.linalg.norm(singx_grad(x_opt)):.3e}")
print(f"Iteraciones = {len(hist['f'])-1}")
print(f"Δ final     = {hist['delta'][-1]:.3e}")   # esperado: pequeño

f(x0)       = 122.0000
||grad(x0)||= 217.2648
rang(H(x0)) = 3

k=  0  f=1.452831e+01  ||g||=5.708e+01  Δ=2.000e+00  ρ=9.655e-01
k=  1  f=5.715363e+00  ||g||=2.474e+01  Δ=2.000e+00  ρ=1.169e+00
k=  2  f=3.064529e+00  ||g||=1.504e+01  Δ=2.000e+00  ρ=1.006e+00
k=  3  f=2.365070e+00  ||g||=7.878e+00  Δ=2.000e+00  ρ=1.042e+00
k=  4  f=1.965128e+00  ||g||=8.918e+00  Δ=2.000e+00  ρ=1.002e+00
k= 20  f=4.041933e-01  ||g||=2.628e+00  Δ=2.000e+00  ρ=1.000e+00
k= 40  f=1.449046e-01  ||g||=1.390e+00  Δ=2.000e+00  ρ=1.000e+00
k= 60  f=6.523437e-02  ||g||=8.270e-01  Δ=2.000e+00  ρ=1.000e+00
k= 80  f=3.471977e-02  ||g||=5.407e-01  Δ=2.000e+00  ρ=1.000e+00
k=100  f=2.081951e-02  ||g||=3.801e-01  Δ=2.000e+00  ρ=1.000e+00
k=120  f=1.360494e-02  ||g||=2.822e-01  Δ=2.000e+00  ρ=1.000e+00
k=140  f=9.474018e-03  ||g||=2.184e-01  Δ=2.000e+00  ρ=1.000e+00
k=160  f=6.924118e-03  ||g||=1.746e-01  Δ=2.000e+00  ρ=1.000e+00
k=180  f=5.254941e-03  ||g||=1.432e-01  Δ=2.000e+00  ρ=1.000e+00
k=200  f=4.109902e-03  ||g|